Importation des bibliothèque

In [2]:
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt 
from biopandas.pdb import PandasPdb
import nglview as nv
from scipy import stats

In [3]:
def view_structure(file_pdb):
    # 1. Charger le fichier généré
    view = nv.show_file(file_pdb)
    # 2. Vider les représentations par défaut
    view.clear_representations()
    # 3. Afficher toute la molécule de manière fine
    view.add_representation('licorice', selection='all', color='element')
    # 4. Mettre en surbrillance spéciale les atomes de Phosphore (les "noeuds" des liaisons phosphodiester)
    view.add_representation('spacefill', selection='_P', color='orange', radius=0.8)
    # 5. Rajouter un "tube" qui suit et met en évidence de façon continue le squelette phosphodiester (backbone)
    view.add_representation('tube', selection='backbone', color='red', radius=0.2)
    
    # # 6. Ajout d'un dégradé de couleur pour voir l'orientation 5' -> 3'
    # # Le schéma 'resindex' colore du bleu (début/5') au rouge (fin/3')
    # view.add_representation('cartoon', selection='nucleic', color='resindex')
    
    # 7. Centrer la vue
    view.center()
    # Afficher l'interface
    return view

In [7]:
view_structure("final_arn.pdb")

NGLWidget()

In [ ]:
ppdb_df =  PandasPdb().read_pdb("fichier_arn/initial_optimized.pdb")
atom_df = ppdb_df.df["ATOM"]
atom_df.head(20)

In [ ]:
from Bio.PDB import Structure, Model, Chain, Residue, Atom, PDBIO

def generer_structure_depart_c3(seq, distance_moy=5.8, output="initial_c3.pdb"):
    # Création de la hiérarchie
    structure = Structure.Structure("RNA")
    model = Model.Model(0)
    chain = Chain.Chain("A")
    
    for i, nt in enumerate(seq):
        # Création du résidu (nucléotide)
        res = Residue.Residue((" ", i+1, " "), nt, " ")
        # Création de l'atome C3' à sa position x
        atom = Atom.Atom("C3'", [i * distance_moy, 0, 0], 0, 0, " ", "C3'", i+1, "C")
        res.add(atom)
        chain.add(res)
        
    model.add(chain)
    structure.add(model)
    
    # Sauvegarde
    io = PDBIO()
    io.set_structure(structure)
    io.save(output)


def voir_configuration_c3(fichier_pdb):
    """
    Visualisation NGL adaptée au modèle C3'-seulement.
    """
    view = nv.show_file(fichier_pdb)
    view.clear_representations()
    view.add_representation("ball+stick", selection="all", color="cyan", radius=0.25)
    view.center()
    return view


# Exemple rapide :
generer_structure_depart_c3(seq="ACGU"*30, distance_moy=5.8)
voir_configuration_c3("fichier_arn/initial_c3_beads.pdb")

In [ ]:
import matplotlib.pyplot as plt
import scipy.stats as stats

# Création du graphique Q-Q
df = pd.read_csv("distances_1-5.csv")
data = df["Distance"]
stats.probplot(data, dist="norm", plot=plt)
plt.title("Graphique Q-Q pour vérifier la normalité")
plt.show()

In [7]:
view_structure("resultat/optimized_bs_dfire_1775210434_full_atom.pdb")

ValueError: you must provide file extension if using file-like object or text content